## Data Prep

In [3]:
import pandas as pd

In [51]:
df = pd.read_csv("311_Service_Requests_from_2020_to_Present_20260602.csv")

In [52]:
# drop redundant columns
df = df.drop(columns=['City','Closed Date','Agency','Agency Name','Problem (formerly Complaint Type)',
                      'Additional Details','Incident Address','Street Name','Cross Street 1','Cross Street 2',
                      'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'Landmark','Facility Type',
                      'Status','Due Date','Resolution Description','Resolution Action Updated Date','Community Board',
                      'Police Precinct','BBL','Open Data Channel Type', 'Park Facility Name','Park Borough','Vehicle Type',
                      'Taxi Company Borough','Taxi Pick Up Location','Bridge Highway Name','Bridge Highway Segment','Bridge Highway Direction',
                      'Road Ramp','Bridge Highway Segment','Location','X Coordinate (State Plane)','Y Coordinate (State Plane)'])

In [53]:
# drop null values
df = df.dropna()

In [54]:
# standardize values
df['Location Type'] = df['Location Type'].str.strip()
df['Location Type'] = df['Location Type'].str.replace('1-2 FamilyDwelling', '1-2 Family Dwelling')
df['Location Type'] = df['Location Type'].str.replace('3+Family Apt.', '3+ Family Apt.')
df['Location Type'] = df['Location Type'].str.replace('3+ Family Apt.', '3+ Family Apt. Building')
df['Location Type'] = df['Location Type'].str.replace('Catch Basin/Sewer', 'Catch Basin or Sewer')
df['Location Type'] = df['Location Type'].str.replace('Parking Lot/Garage', 'Parking Lot or Garage')
df['Location Type'] = df['Location Type'].str.replace('Day Care/Nursery', 'Day Care or Nursery')

df = df[df['Borough'] != 'Unspecified']
df = df.dropna(subset=['Borough'])

In [55]:
# parse month and season
df['Created Date'] = pd.to_datetime(df['Created Date'])
df['month'] = df['Created Date'].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

In [56]:
df.groupby('Borough').size()

Borough
BRONX            23173
BROOKLYN         55083
MANHATTAN        37558
QUEENS           23347
STATEN ISLAND     4170
dtype: int64

In [57]:
df.head()

,Unique Key,Created Date,Problem Detail (formerly Descriptor),Location Type,Incident Zip,Council District,Borough,Latitude,Longitude,month,season
0,45287002,2020-01-01 03:24:58,Rat Sighting,3+ Family Apt. Building Building,10031.0,7.0,MANHATTAN,40.826844,-73.947379,1,Winter
1,45288587,2020-01-01 06:57:23,Rat Sighting,Commercial Building,10314.0,50.0,STATEN ISLAND,40.612035,-74.137318,1,Winter
2,45289761,2020-01-01 07:04:51,Rat Sighting,3+ Family Apt. Building Building,10026.0,9.0,MANHATTAN,40.800206,-73.952489,1,Winter
3,45286905,2020-01-01 10:35:31,Rat Sighting,3+ Family Apt. Building Building,11238.0,35.0,BROOKLYN,40.672620,-73.965792,1,Winter
4,45286804,2020-01-01 11:24:55,Rat Sighting,Street Area,11211.0,34.0,BROOKLYN,40.715430,-73.938752,1,Winter


## Locating Neighborhood

In [1]:
!pip install tqdm

In [ ]:
import pandas as pd
import googlemaps
from tqdm.notebook import tqdm

gmaps = googlemaps.Client(key='')
tqdm.pandas()

In [58]:
# Step 1 — reload clean data
print(df['Borough'].value_counts())  # confirm all 5 boroughs are here

# Step 2 — recreate unique_coords from the full dataset
unique_coords = df[['Latitude', 'Longitude']].drop_duplicates().dropna()
print(f"Total unique coordinates: {len(unique_coords)}")

Borough
BROOKLYN         55083
MANHATTAN        37558
QUEENS           23347
BRONX            23173
STATEN ISLAND     4170
Name: count, dtype: int64
Total unique coordinates: 70402


In [59]:
import time

def get_neighborhood(lat, lon):
    try:
        results = gmaps.reverse_geocode((lat, lon))
        for result in results:
            for component in result['address_components']:
                if 'neighborhood' in component['types']:
                    return component['long_name']
                elif 'sublocality_level_2' in component['types']:
                    return component['long_name']
                elif 'sublocality_level_3' in component['types']:
                    return component['long_name']
        for result in results:
            for component in result['address_components']:
                if 'sublocality_level_1' in component['types']:
                    return component['long_name']
        return 'Unknown'
    except Exception as e:
        print(f"Error at ({lat}, {lon}): {e}")
        return 'Unknown'

In [60]:
# Flushing, Queens
test_result = get_neighborhood(40.7675, -73.8330)
print(test_result)  # should return 'Flushing'

Flushing


In [61]:
# process in batches and save progress
batch_size = 1000
results_list = []

for i in tqdm(range(0, len(unique_coords), batch_size)):
    batch = unique_coords.iloc[i:i+batch_size]
    batch = batch.copy()
    batch['neighborhood'] = batch.apply(
        lambda row: get_neighborhood(row['Latitude'], row['Longitude']), axis=1
    )
    results_list.append(batch)
    
    # save progress after every batch
    pd.concat(results_list).to_csv('neighborhoods_progress.csv', index=False)
    print(f"Saved batch {i//batch_size + 1} — {min(i+batch_size, len(unique_coords))}/{len(unique_coords)} done")
    time.sleep(0.5)  # small delay to avoid hitting API too hard

# combine all batches
unique_coords = pd.concat(results_list)
print("All done!")

  0%|          | 0/71 [00:00<?, ?it/s]

Saved batch 1 — 1000/70402 done
Saved batch 2 — 2000/70402 done
Saved batch 3 — 3000/70402 done
Saved batch 4 — 4000/70402 done
Saved batch 5 — 5000/70402 done
Saved batch 6 — 6000/70402 done
Saved batch 7 — 7000/70402 done
Saved batch 8 — 8000/70402 done
Saved batch 9 — 9000/70402 done
Saved batch 10 — 10000/70402 done
Saved batch 11 — 11000/70402 done
Saved batch 12 — 12000/70402 done
Saved batch 13 — 13000/70402 done
Saved batch 14 — 14000/70402 done
Saved batch 15 — 15000/70402 done
Saved batch 16 — 16000/70402 done
Saved batch 17 — 17000/70402 done
Saved batch 18 — 18000/70402 done
Saved batch 19 — 19000/70402 done
Saved batch 20 — 20000/70402 done
Saved batch 21 — 21000/70402 done
Saved batch 22 — 22000/70402 done
Saved batch 23 — 23000/70402 done
Saved batch 24 — 24000/70402 done
Saved batch 25 — 25000/70402 done
Saved batch 26 — 26000/70402 done
Saved batch 27 — 27000/70402 done
Saved batch 28 — 28000/70402 done
Saved batch 29 — 29000/70402 done
Saved batch 30 — 30000/70402 don

In [62]:
df = df.merge(unique_coords[['Latitude', 'Longitude', 'neighborhood']], 
              on=['Latitude', 'Longitude'], 
              how='left')

In [63]:
df

,Unique Key,Created Date,Problem Detail (formerly Descriptor),Location Type,Incident Zip,Council District,Borough,Latitude,Longitude,month,season,neighborhood
0,45287002,2020-01-01 03:24:58,Rat Sighting,3+ Family Apt. Building Building,10031.0,7.0,MANHATTAN,40.826844,-73.947379,1,Winter,Hamilton Heights
1,45288587,2020-01-01 06:57:23,Rat Sighting,Commercial Building,10314.0,50.0,STATEN ISLAND,40.612035,-74.137318,1,Winter,Castleton Corners
2,45289761,2020-01-01 07:04:51,Rat Sighting,3+ Family Apt. Building Building,10026.0,9.0,MANHATTAN,40.800206,-73.952489,1,Winter,Central Harlem
3,45286905,2020-01-01 10:35:31,Rat Sighting,3+ Family Apt. Building Building,11238.0,35.0,BROOKLYN,40.672620,-73.965792,1,Winter,Prospect Heights
4,45286804,2020-01-01 11:24:55,Rat Sighting,Street Area,11211.0,34.0,BROOKLYN,40.715430,-73.938752,1,Winter,East Williamsburg
...,...,...,...,...,...,...,...,...,...,...,...,...
143326,69179237,2026-05-30 22:52:56,Rat Sighting,3+ Family Apt. Building Building,11249.0,34.0,BROOKLYN,40.704468,-73.964589,5,Spring,Williamsburg
143327,69175230,2026-05-30 22:54:20,Rat Sighting,3+ Family Apt. Building Building,11249.0,34.0,BROOKLYN,40.704819,-73.963334,5,Spring,Williamsburg
143328,69180635,2026-05-31 00:19:23,Rat Sighting,Sidewalk,10128.0,5.0,MANHATTAN,40.784054,-73.948753,5,Spring,Yorkville
143329,69175228,2026-05-31 00:46:27,Rat Sighting,3+ Family Apt. Building Building,11216.0,36.0,BROOKLYN,40.681391,-73.946859,5,Spring,Bedford-Stuyvesant


## Modeling

In [66]:
# Aggregate Data by neighborhood
cluster_df = df.groupby(['neighborhood', 'Borough', 'season']).agg(
    sighting_count=('Unique Key', 'count'),
).reset_index()

print(cluster_df.shape)
print(cluster_df.head())

(1177, 4)
    neighborhood    Borough  season  sighting_count
0       Allerton      BRONX    Fall             130
1       Allerton      BRONX  Spring             122
2       Allerton      BRONX  Summer             143
3       Allerton      BRONX  Winter             102
4  Alphabet City  MANHATTAN    Fall             217


In [67]:
# Encode categorical columns
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
cluster_df['Borough_encoded']  = le.fit_transform(cluster_df['Borough'])
cluster_df['season_encoded']   = le.fit_transform(cluster_df['season'])
cluster_df['neighborhood_encoded'] = le.fit_transform(cluster_df['neighborhood'])

In [68]:
# Scale features
features = ['Borough_encoded', 'season_encoded', 'neighborhood_encoded', 'sighting_count']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df[features])

In [69]:
# Train K-means
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_scaled)
cluster_df['cluster'] = kmeans.labels_

print(cluster_df['cluster'].value_counts())

cluster
0    569
1    553
2     55
Name: count, dtype: int64


In [70]:
# Label clusters meaningfully
cluster_means = cluster_df.groupby('cluster')['sighting_count'].mean().sort_values()
label_map = {cluster_means.index[0]: 'Low', 
             cluster_means.index[1]: 'Medium', 
             cluster_means.index[2]: 'High'}
cluster_df['risk_level'] = cluster_df['cluster'].map(label_map)
print(cluster_df[['neighborhood', 'Borough', 'risk_level', 'sighting_count']].head(10))

    neighborhood        Borough risk_level  sighting_count
0       Allerton          BRONX     Medium             130
1       Allerton          BRONX     Medium             122
2       Allerton          BRONX     Medium             143
3       Allerton          BRONX     Medium             102
4  Alphabet City      MANHATTAN     Medium             217
5  Alphabet City      MANHATTAN     Medium             202
6  Alphabet City      MANHATTAN     Medium             257
7  Alphabet City      MANHATTAN        Low             127
8       Annadale  STATEN ISLAND        Low               7
9       Annadale  STATEN ISLAND        Low               8


In [71]:
cluster_df.to_csv('rat_clustered.csv', index=False)